# NV色心ODMR实验控制界面
负责智能体：【@S 软件工程师】
功能：NV色心ODMR扫描的交互式控制与数据可视化
依赖：tcp_server.py, sg386_control.py

【知识来源】：NV项目物理参数 + PYNQ Jupyter使用手册

In [ ]:
# ============================================================
# NV ODMR控制界面初始化
# ============================================================

import sys
sys.path.append('/home/xilinx/pynq/drivers')

from tcp_server import ODMRServer
from sg386_control import SG386Controller
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

print('【@S 软件工程师】NV ODMR控制界面加载完成')

## 1. 硬件连接初始化

In [ ]:
# ============================================================
# 硬件连接初始化
# ============================================================

# ODMR TCP服务器初始化 (PL端控制)
odmr_server = ODMRServer(host='192.168.1.10', port=5000)
print('【@S 软件工程师】ODMR服务器已初始化')

# SG386微波源初始化 (USB控制)
try:
    sg386 = SG386Controller(port='/dev/ttyUSB0')
    print('【@S 软件工程师】SG386微波源已连接')
    
    # 自检
    ok, info = sg386.self_test()
    if ok:
        print(f'【@S 软件工程师】SG386自检通过: {info}')
    else:
        print(f'【@S 软件工程师】SG386自检警告: {info}')
        
except Exception as e:
    print(f'【@S 软件工程师】SG386连接失败: {e}')
    sg386 = None

## 2. 扫描参数配置

In [ ]:
# ============================================================
# 扫描参数配置面板
# ============================================================

# 频率扫描参数
freq_start = widgets.FloatText(
    value=2.80,  # GHz
    description='起始频率(GHz):',
    style={'description_width': '120px'}
)

freq_stop = widgets.FloatText(
    value=2.95,  # GHz
    description='终止频率(GHz):',
    style={'description_width': '120px'}
)

freq_step = widgets.FloatText(
    value=0.5,  # MHz
    description='频率步进(MHz):',
    style={'description_width': '120px'}
)

# 积分时间参数
integ_time = widgets.IntText(
    value=1000,  # ns
    description='积分时间(ns):',
    style={'description_width': '120px'}
)

# 微波源参数
mw_power = widgets.FloatText(
    value=-10.0,  # dBm
    description='微波功率(dBm):',
    style={'description_width': '120px'}
)

# 显示参数面板
display(widgets.VBox([
    widgets.HTML('<h3>【@S 软件工程师】频率扫描参数</h3>'),
    freq_start,
    freq_stop,
    freq_step,
    widgets.HTML('<h3>【@S 软件工程师】积分时间</h3>'),
    integ_time,
    widgets.HTML('<h3>【@S 软件工程师】微波源参数</h3>'),
    mw_power
]))

## 3. 扫描控制函数

In [ ]:
# ============================================================
# ODMR扫描控制函数
# ============================================================

def setup_scan():
    """【@S 软件工程师】配置扫描参数"""
    # 转换单位
    f_start = int(freq_start.value * 1e9)  # GHz -> Hz
    f_stop = int(freq_stop.value * 1e9)   # GHz -> Hz
    f_step = int(freq_step.value * 1e6)    # MHz -> Hz
    t_integ = int(integ_time.value)         # ns
    
    # 计算等待时间 (确保积分完成)
    wait_time = t_integ * 10  # ns
    
    # 设置PL端扫描参数
    success = odmr_server.axi.set_scan_params(
        freq_start=f_start,
        freq_stop=f_stop,
        freq_step=f_step,
        integ_time=t_integ,
        wait_time=wait_time
    )
    
    # 设置微波源功率
    if sg386:
        sg386.set_power(mw_power.value)
        
    if success:
        print(f'【@S 软件工程师】扫描参数配置完成')
        print(f'  频率范围: {f_start/1e9:.4f} - {f_stop/1e9:.4f} GHz')
        print(f'  频率步进: {f_step/1e6:.2f} MHz')
        print(f'  积分时间: {t_integ} ns')
    else:
        print('【@S 软件工程师】扫描参数配置失败')
    
    return success

def start_scan():
    """【@S 软件工程师】启动ODMR扫描"""
    success = odmr_server.axi.start_scan()
    if success:
        odmr_server.start_data_stream()  # 启动数据流
        print('【@S 软件工程师】ODMR扫描已启动')
    return success

def stop_scan():
    """【@S 软件工程师】停止ODMR扫描"""
    odmr_server.stop_data_stream()  # 停止数据流
    success = odmr_server.axi.stop_scan()
    if success:
        print('【@S 软件工程师】ODMR扫描已停止')
    return success

def get_data():
    """【@S 软件工程师】获取扫描数据"""
    data = odmr_server.scan_data_buffer
    print(f'【@S 软件工程师】获取到 {len(data)} 个数据点')
    return data

def plot_odmr():
    """【@S 软件工程师】绘制ODMR曲线"""
    data = get_data()
    if len(data) < 10:
        print('【@S 软件工程师】数据点不足，无法绘图')
        return
    
    # 提取频率和光强
    freqs = np.array([d['frequency'] for d in data])
    intensities = np.array([d['intensity'] for d in data])
    
    # 归一化光强
    intensities = intensities / intensities.max()
    
    # 绘图
    plt.figure(figsize=(10, 6))
    plt.plot(freqs/1e9, intensities, 'b.-')
    plt.xlabel('频率 (GHz)')
    plt.ylabel('归一化光强')
    plt.title('【@S 软件工程师】NV色心ODMR谱线')
    plt.grid(True)
    plt.show()

print('【@S 软件工程师】扫描控制函数已定义')

## 4. 一键控制面板

In [ ]:
# ============================================================
# 一键控制面板
# ============================================================

# 定义按钮
btn_setup = widgets.Button(
    description='配置参数',
    button_style='primary',
    icon='cog'
)

btn_start = widgets.Button(
    description='启动扫描',
    button_style='success',
    icon='play'
)

btn_stop = widgets.Button(
    description='停止扫描',
    button_style='danger',
    icon='stop'
)

btn_plot = widgets.Button(
    description='绘制曲线',
    button_style='info',
    icon='line-chart'
)

# 绑定回调函数
btn_setup.on_click(lambda b: setup_scan())
btn_start.on_click(lambda b: start_scan())
btn_stop.on_click(lambda b: stop_scan())
btn_plot.on_click(lambda b: plot_odmr())

# 显示控制面板
display(widgets.HBox([btn_setup, btn_start, btn_stop, btn_plot]))

## 5. 实时数据监控

In [ ]:
# ============================================================
# 实时数据监控
# ============================================================

# 状态显示
status_label = widgets.HTML(
    value='<b>【@S 软件工程师】扫描状态: 待机</b>'
)

data_count = widgets.IntText(
    value=0,
    description='数据点数:',
    disabled=True,
    style={'description_width': '80px'}
)

progress = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description='进度:',
    bar_style='info',
    style={'description_width': '50px'},
    layout={'width': '300px'}
)

display(widgets.VBox([status_label, data_count, progress]))

## 6. 微波源快速控制

In [ ]:
# ============================================================
# 微波源快速控制
# ============================================================

def set_mw_freq(freq_ghz):
    """【@S 软件工程师】快速设置微波频率"""
    if sg386:
        freq_hz = freq_ghz * 1e9
        success, resp = sg386.set_frequency(freq_hz)
        if success:
            print(f'【@S 软件工程师】微波频率设置为 {freq_ghz:.4f} GHz')
        return success
    else:
        print('【@S 软件工程师】SG386未连接')
        return False

def set_mw_power(power_dbm):
    """【@S 软件工程师】快速设置微波功率"""
    if sg386:
        success, resp = sg386.set_power(power_dbm)
        if success:
            print(f'【@S 软件工程师】微波功率设置为 {power_dbm:.1f} dBm')
        return success
    else:
        print('【@S 软件工程师】SG386未连接')
        return False

def mw_output_on():
    """【@S 软件工程师】开启微波输出"""
    if sg386:
        sg386.output_on()
        print('【@S 软件工程师】微波输出已开启')

def mw_output_off():
    """【@S 软件工程师】关闭微波输出"""
    if sg386:
        sg386.output_off()
        print('【@S 软件工程师】微波输出已关闭')

# NV色心典型频率快捷设置
print('【@S 软件工程师】NV色心典型共振频率:')
print('  - NV-0: 2.865 GHz')
print('  - NV-:  2.885 GHz')
print('  - 可使用 set_mw_freq(2.87) 快速设置')

---

## 使用说明

1. **配置参数**: 设置扫描频率范围、积分时间、微波功率
2. **点击"配置参数"**: 将参数下发到PL端
3. **点击"启动扫描"**: 开始ODMR扫描
4. **点击"绘制曲线"**: 查看ODMR谱线
5. **点击"停止扫描"**: 结束扫描

**快捷命令**:
- `set_mw_freq(2.87)` - 设置微波频率为2.87 GHz
- `set_mw_power(-10)` - 设置微波功率为-10 dBm
- `mw_output_on()` - 开启微波输出
- `mw_output_off()` - 关闭微波输出

In [ ]:
print('【@S 软件工程师】NV ODMR控制界面已就绪')
print('【@S 软件工程师】请先配置参数，然后点击"配置参数"开始实验')